# NBA Game Prediction

In [1]:
import pandas as pd

df = pd.read_csv("nba_games.csv")

In [2]:
df = df.sort_values("date")
df = df.reset_index(drop=True)
df

,Unnamed: 0,mp,fg,fga,fg%,3p,3pa,3p%,ft,fta,...,pf_max_opp,pts_max_opp,gmsc_max_opp,+/-_max_opp,team_opp,total_opp,home_opp,season,date,won
0,865,240.0,38.0,81.0,0.469,9.0,29.0,0.310,24.0,31.0,...,5.0,33.0,24.3,17.0,LAC,116,0,2021,2020-12-22,False
1,4996,240.0,37.0,99.0,0.374,10.0,33.0,0.303,15.0,23.0,...,3.0,26.0,21.0,32.0,BRK,125,1,2021,2020-12-22,False
2,4997,240.0,42.0,92.0,0.457,15.0,35.0,0.429,26.0,32.0,...,4.0,20.0,16.1,9.0,GSW,99,0,2021,2020-12-22,True
3,864,240.0,44.0,93.0,0.473,14.0,40.0,0.350,14.0,19.0,...,5.0,22.0,19.7,14.0,LAL,109,1,2021,2020-12-22,True
4,5118,265.0,45.0,102.0,0.441,10.0,31.0,0.323,24.0,31.0,...,6.0,29.0,29.3,5.0,DEN,122,1,2021,2020-12-23,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7007,3450,240.0,35.0,85.0,0.412,12.0,34.0,0.353,12.0,15.0,...,5.0,34.0,29.4,25.0,OKC,124,1,2025,2025-05-28,False
7008,148,240.0,30.0,74.0,0.405,10.0,30.0,0.333,24.0,29.0,...,5.0,32.0,26.1,26.0,NYK,111,1,2025,2025-05-29,False
7009,149,240.0,44.0,89.0,0.494,8.0,29.0,0.276,15.0,22.0,...,4.0,23.0,17.3,8.0,IND,94,0,2025,2025-05-29,True
7010,1508,240.0,41.0,86.0,0.477,9.0,32.0,0.281,17.0,26.0,...,5.0,31.0,26.3,25.0,IND,125,1,2025,2025-05-31,False


## Build the target: did the team win its *next* game?

In [3]:
def add_target(team):
    team["target"] = team["won"].shift(-1)
    return team

df = df.groupby("team", group_keys=False).apply(add_target)

C:\Users\User\AppData\Local\Temp\ipykernel_9484\1127067056.py:5: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby("team", group_keys=False).apply(add_target)


In [4]:
df[df["team"] == "WAS"]

,Unnamed: 0,mp,fg,fga,fg%,3p,3pa,3p%,ft,fta,...,pts_max_opp,gmsc_max_opp,+/-_max_opp,team_opp,total_opp,home_opp,season,date,won,target
11,3056,240.0,39.0,85.0,0.459,13.0,27.0,0.481,16.0,23.0,...,29.0,23.6,33.0,PHI,113,1,2021,2020-12-23,False,False
41,741,240.0,46.0,95.0,0.484,13.0,37.0,0.351,15.0,28.0,...,25.0,20.1,16.0,ORL,130,0,2021,2020-12-26,False,False
77,737,240.0,47.0,97.0,0.485,8.0,32.0,0.250,11.0,13.0,...,26.0,16.4,28.0,ORL,120,0,2021,2020-12-27,False,False
103,5921,240.0,37.0,84.0,0.440,10.0,37.0,0.270,23.0,26.0,...,23.0,21.2,14.0,CHI,115,0,2021,2020-12-29,False,False
130,1525,240.0,42.0,84.0,0.500,14.0,29.0,0.483,32.0,41.0,...,28.0,31.3,23.0,CHI,133,0,2021,2020-12-31,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6823,2277,240.0,35.0,87.0,0.402,11.0,37.0,0.297,23.0,34.0,...,21.0,22.4,10.0,TOR,112,0,2025,2025-03-24,False,True
6848,2436,240.0,48.0,95.0,0.505,18.0,44.0,0.409,5.0,12.0,...,22.0,21.4,10.0,PHI,114,1,2025,2025-03-26,True,False
6868,6313,240.0,38.0,85.0,0.447,11.0,30.0,0.367,22.0,26.0,...,29.0,26.4,31.0,IND,162,0,2025,2025-03-27,False,False
6895,869,240.0,37.0,85.0,0.435,9.0,34.0,0.265,29.0,33.0,...,20.0,20.0,13.0,BRK,115,0,2025,2025-03-29,False,False


A team's last game of the season has no "next game" to predict, so its `target` comes out `NaN`. Mark those rows with `2` instead of leaving them null, and cast the column to int.

In [5]:
df.loc[pd.isnull(df["target"]), "target"] = 2
df["target"] = df["target"].astype(int)

In [6]:
df["won"].value_counts()

won
False    3506
True     3506
Name: count, dtype: int64

In [7]:
df["target"].value_counts()

target
1    3491
0    3491
2      30
Name: count, dtype: int64

## Drop any column that has nulls anywhere

In [8]:
nulls = pd.isnull(df).sum()
nulls = nulls[nulls > 0]
valid_columns = df.columns[~df.columns.isin(nulls.index)]
valid_columns

Index(['Unnamed: 0', 'mp', 'fg', 'fga', 'fg%', '3p', '3pa', '3p%', 'ft', 'fta',
       'ft%', 'orb', 'drb', 'trb', 'ast', 'stl', 'blk', 'tov', 'pf', 'pts',
       'fg_max', 'fga_max', 'fg%_max', '3p_max', '3pa_max', '3p%_max',
       'ft_max', 'fta_max', 'ft%_max', 'orb_max', 'drb_max', 'trb_max',
       'ast_max', 'stl_max', 'blk_max', 'tov_max', 'pf_max', 'pts_max',
       'gmsc_max', '+/-_max', 'team', 'total', 'home', 'mp_opp', 'fg_opp',
       'fga_opp', 'fg%_opp', '3p_opp', '3pa_opp', '3p%_opp', 'ft_opp',
       'fta_opp', 'ft%_opp', 'orb_opp', 'drb_opp', 'trb_opp', 'ast_opp',
       'stl_opp', 'blk_opp', 'tov_opp', 'pf_opp', 'pts_opp', 'fg_max_opp',
       'fga_max_opp', 'fg%_max_opp', '3p_max_opp', '3pa_max_opp',
       '3p%_max_opp', 'ft_max_opp', 'fta_max_opp', 'ft%_max_opp',
       'orb_max_opp', 'drb_max_opp', 'trb_max_opp', 'ast_max_opp',
       'stl_max_opp', 'blk_max_opp', 'tov_max_opp', 'pf_max_opp',
       'pts_max_opp', 'gmsc_max_opp', '+/-_max_opp', 'team_opp', 'tota

In [9]:
df = df[valid_columns].copy()
df

,Unnamed: 0,mp,fg,fga,fg%,3p,3pa,3p%,ft,fta,...,pts_max_opp,gmsc_max_opp,+/-_max_opp,team_opp,total_opp,home_opp,season,date,won,target
0,865,240.0,38.0,81.0,0.469,9.0,29.0,0.310,24.0,31.0,...,33.0,24.3,17.0,LAC,116,0,2021,2020-12-22,False,1
1,4996,240.0,37.0,99.0,0.374,10.0,33.0,0.303,15.0,23.0,...,26.0,21.0,32.0,BRK,125,1,2021,2020-12-22,False,0
2,4997,240.0,42.0,92.0,0.457,15.0,35.0,0.429,26.0,32.0,...,20.0,16.1,9.0,GSW,99,0,2021,2020-12-22,True,1
3,864,240.0,44.0,93.0,0.473,14.0,40.0,0.350,14.0,19.0,...,22.0,19.7,14.0,LAL,109,1,2021,2020-12-22,True,1
4,5118,265.0,45.0,102.0,0.441,10.0,31.0,0.323,24.0,31.0,...,29.0,29.3,5.0,DEN,122,1,2021,2020-12-23,True,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7007,3450,240.0,35.0,85.0,0.412,12.0,34.0,0.353,12.0,15.0,...,34.0,29.4,25.0,OKC,124,1,2025,2025-05-28,False,2
7008,148,240.0,30.0,74.0,0.405,10.0,30.0,0.333,24.0,29.0,...,32.0,26.1,26.0,NYK,111,1,2025,2025-05-29,False,1
7009,149,240.0,44.0,89.0,0.494,8.0,29.0,0.276,15.0,22.0,...,23.0,17.3,8.0,IND,94,0,2025,2025-05-29,True,0
7010,1508,240.0,41.0,86.0,0.477,9.0,32.0,0.281,17.0,26.0,...,31.0,26.3,25.0,IND,125,1,2025,2025-05-31,False,2


## Feature selection

Scale the features, impute any remaining gaps, then use forward sequential feature selection with a `RidgeClassifier` to pick the 30 most useful predictors. `TimeSeriesSplit` is used for cross-validation since this is time-ordered data.

In [10]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.linear_model import RidgeClassifier
from sklearn.preprocessing import MinMaxScaler
from sklearn.impute import SimpleImputer

removed_columns = ["season", "date", "won", "target", "team", "team_opp"]
selected_columns = df.columns[~df.columns.isin(removed_columns)]

scaler = MinMaxScaler()
df[selected_columns] = scaler.fit_transform(df[selected_columns])

imputer = SimpleImputer(strategy="mean")
X = imputer.fit_transform(df[selected_columns])

rr = RidgeClassifier(alpha=1)
split = TimeSeriesSplit(n_splits=3)
sfs = SequentialFeatureSelector(rr, n_features_to_select=30, direction="forward", cv=split)

sfs.fit(X, df["target"])

,estimator,RidgeClassifier(alpha=1)
,n_features_to_select,30
,tol,None
,direction,'forward'
,scoring,None
,cv,TimeSeriesSpl...est_size=None)
,n_jobs,None
,alpha,1
,fit_intercept,True
,copy_X,True
,max_iter,None


In [11]:
predictors = list(selected_columns[sfs.get_support()])
predictors

['Unnamed: 0',
 'mp',
 'fga',
 '3p',
 'ft',
 'fta',
 'ft%',
 'fg_max',
 'fg%_max',
 '3p%_max',
 'ft%_max',
 'orb_max',
 'pf_max',
 'pts_max',
 'gmsc_max',
 '+/-_max',
 'mp_opp',
 'fga_opp',
 '3p%_opp',
 'ast_opp',
 'blk_opp',
 'tov_opp',
 'pf_opp',
 'ft%_max_opp',
 'drb_max_opp',
 'trb_max_opp',
 'stl_max_opp',
 'tov_max_opp',
 'pts_max_opp',
 '+/-_max_opp']

## Backtesting the model

Walk the model forward season by season: train on everything before a season, predict that season, and stack up the results.

In [12]:
def backtest(data, model, predictors, start=2, step=1):
    all_predictions = []
    seasons = sorted(data["season"].unique())

    for i in range(start, len(seasons), step):
        season = seasons[i]
        train = data[data["season"] < season]
        test = data[data["season"] == season]

        model.fit(train[predictors], train["target"])

        preds = model.predict(test[predictors])
        preds = pd.Series(preds, index=test.index)
        combined = pd.concat([test["target"], preds], axis=1)
        combined.columns = ["actual", "prediction"]

        all_predictions.append(combined)
    return pd.concat(all_predictions)

In [13]:
predictions = backtest(df, rr, predictors)

In [16]:
predictions

,actual,prediction
2066,1,1
2067,1,1
2068,1,1
2069,0,0
2070,1,0
...,...,...
7003,1,1
7004,1,1
7005,0,1
7008,1,1


Rows where `target` is `2` are a team's last game of the season (no next-game result to predict), so drop those before scoring.

In [14]:
from sklearn.metrics import accuracy_score

predictions = predictions[predictions["actual"] != 2]
accuracy_score(predictions["actual"], predictions["prediction"])

0.5541090317331163

In [15]:
predictions["prediction"].value_counts()

prediction
0    2548
1    2368
Name: count, dtype: int64

In [17]:
predictions["actual"].value_counts()

actual
1    2458
0    2458
Name: count, dtype: int64

Baseline: how often does the home team simply win?

In [18]:
df.groupby("home").apply(lambda x: x[x["won"] == 1].shape[0] / x.shape[0])

C:\Users\User\AppData\Local\Temp\ipykernel_9484\3128703651.py:1: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df.groupby("home").apply(lambda x: x[x["won"] == 1].shape[0] / x.shape[0])


home
0.0    0.445807
1.0    0.554193
dtype: float64

## Adding rolling averages

Each game's raw stats are noisy. Rolling 10-game averages per team, per season, tend to carry more predictive signal than a single game's box score.

In [33]:
selected_columns = df.columns.difference([
    "date",
    "won",
    "target",
    "team",
    "team_opp",
    "season"
])

In [34]:
df_rolling = df[list(selected_columns) + ["won", "team", "season"]]

In [32]:
rolling_cols = [f"{col}_10" for col in df_rolling.columns]
df_rolling.columns = rolling_cols
df = pd.concat([df, df_rolling], axis=1)

In [21]:
df

,Unnamed: 0,mp,fg,fga,fg%,3p,3pa,3p%,ft,fta,...,tov_max_opp_10,pf_max_opp_10,pts_max_opp_10,gmsc_max_opp_10,+/-_max_opp_10,total_opp_10,home_opp_10,won_10,team_10,season_10
0,0.123378,0.0,0.390244,0.298246,0.446701,0.259259,0.346154,0.386401,0.523810,0.571429,...,0.363636,0.8,0.355932,0.311321,0.500000,0.515789,0.0,False,LAL,2021
1,0.712594,0.0,0.365854,0.614035,0.205584,0.296296,0.423077,0.374793,0.309524,0.408163,...,0.181818,0.4,0.237288,0.249057,0.714286,0.610526,1.0,False,GSW,2021
2,0.712737,0.0,0.487805,0.491228,0.416244,0.481481,0.461538,0.583748,0.571429,0.591837,...,0.272727,0.6,0.135593,0.156604,0.385714,0.336842,0.0,True,BRK,2021
3,0.123235,0.0,0.536585,0.508772,0.456853,0.444444,0.557692,0.452736,0.285714,0.326531,...,0.272727,0.8,0.169492,0.224528,0.457143,0.442105,1.0,True,LAC,2021
4,0.729996,0.5,0.560976,0.666667,0.375635,0.296296,0.384615,0.407960,0.523810,0.571429,...,0.454545,1.0,0.288136,0.405660,0.328571,0.578947,1.0,True,SAC,2021
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7007,0.492084,0.0,0.317073,0.368421,0.302030,0.370370,0.442308,0.457711,0.238095,0.244898,...,0.272727,0.8,0.372881,0.407547,0.614286,0.600000,1.0,False,MIN,2025
7008,0.021110,0.0,0.195122,0.175439,0.284264,0.296296,0.365385,0.424544,0.523810,0.530612,...,0.272727,0.8,0.338983,0.345283,0.628571,0.463158,1.0,False,IND,2025
7009,0.021252,0.0,0.536585,0.438596,0.510152,0.222222,0.346154,0.330017,0.309524,0.387755,...,0.272727,0.6,0.186441,0.179245,0.371429,0.284211,0.0,True,NYK,2025
7010,0.215091,0.0,0.463415,0.385965,0.467005,0.259259,0.403846,0.338308,0.357143,0.469388,...,0.272727,0.8,0.322034,0.349057,0.614286,0.610526,1.0,False,NYK,2025


In [22]:
df = df.dropna()
df

,Unnamed: 0,mp,fg,fga,fg%,3p,3pa,3p%,ft,fta,...,tov_max_opp_10,pf_max_opp_10,pts_max_opp_10,gmsc_max_opp_10,+/-_max_opp_10,total_opp_10,home_opp_10,won_10,team_10,season_10
0,0.123378,0.0,0.390244,0.298246,0.446701,0.259259,0.346154,0.386401,0.523810,0.571429,...,0.363636,0.8,0.355932,0.311321,0.500000,0.515789,0.0,False,LAL,2021
1,0.712594,0.0,0.365854,0.614035,0.205584,0.296296,0.423077,0.374793,0.309524,0.408163,...,0.181818,0.4,0.237288,0.249057,0.714286,0.610526,1.0,False,GSW,2021
2,0.712737,0.0,0.487805,0.491228,0.416244,0.481481,0.461538,0.583748,0.571429,0.591837,...,0.272727,0.6,0.135593,0.156604,0.385714,0.336842,0.0,True,BRK,2021
3,0.123235,0.0,0.536585,0.508772,0.456853,0.444444,0.557692,0.452736,0.285714,0.326531,...,0.272727,0.8,0.169492,0.224528,0.457143,0.442105,1.0,True,LAC,2021
4,0.729996,0.5,0.560976,0.666667,0.375635,0.296296,0.384615,0.407960,0.523810,0.571429,...,0.454545,1.0,0.288136,0.405660,0.328571,0.578947,1.0,True,SAC,2021
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7007,0.492084,0.0,0.317073,0.368421,0.302030,0.370370,0.442308,0.457711,0.238095,0.244898,...,0.272727,0.8,0.372881,0.407547,0.614286,0.600000,1.0,False,MIN,2025
7008,0.021110,0.0,0.195122,0.175439,0.284264,0.296296,0.365385,0.424544,0.523810,0.530612,...,0.272727,0.8,0.338983,0.345283,0.628571,0.463158,1.0,False,IND,2025
7009,0.021252,0.0,0.536585,0.438596,0.510152,0.222222,0.346154,0.330017,0.309524,0.387755,...,0.272727,0.6,0.186441,0.179245,0.371429,0.284211,0.0,True,NYK,2025
7010,0.215091,0.0,0.463415,0.385965,0.467005,0.259259,0.403846,0.338308,0.357143,0.469388,...,0.272727,0.8,0.322034,0.349057,0.614286,0.610526,1.0,False,NYK,2025


## Bringing in next-game and opponent context

For each row, figure out the team's *next* game (opponent, date, home/away), then merge in that opponent's most recent rolling stats — this gives the model a look at both sides of the matchup.

In [23]:
def shift_col(team, col_name):
    next_col = team[col_name].shift(-1)
    return next_col

def add_col(df, col_name):
    return df.groupby("team", group_keys=False).apply(lambda x: shift_col(x, col_name))

df["home_next"] = add_col(df, "home")
df["team_opp_next"] = add_col(df, "team_opp")
df["date_next"] = add_col(df, "date")

C:\Users\User\AppData\Local\Temp\ipykernel_9484\1688004556.py:6: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return df.groupby("team", group_keys=False).apply(lambda x: shift_col(x, col_name))
C:\Users\User\AppData\Local\Temp\ipykernel_9484\1688004556.py:6: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return df.groupby("team", group_keys=False).apply(lambda x: shift_col(x, col_name))
C:\Users\User\AppData\Local\Te

In [24]:
df

,Unnamed: 0,mp,fg,fga,fg%,3p,3pa,3p%,ft,fta,...,gmsc_max_opp_10,+/-_max_opp_10,total_opp_10,home_opp_10,won_10,team_10,season_10,home_next,team_opp_next,date_next
0,0.123378,0.0,0.390244,0.298246,0.446701,0.259259,0.346154,0.386401,0.523810,0.571429,...,0.311321,0.500000,0.515789,0.0,False,LAL,2021,1.0,DAL,2020-12-25
1,0.712594,0.0,0.365854,0.614035,0.205584,0.296296,0.423077,0.374793,0.309524,0.408163,...,0.249057,0.714286,0.610526,1.0,False,GSW,2021,0.0,MIL,2020-12-25
2,0.712737,0.0,0.487805,0.491228,0.416244,0.481481,0.461538,0.583748,0.571429,0.591837,...,0.156604,0.385714,0.336842,0.0,True,BRK,2021,0.0,BOS,2020-12-25
3,0.123235,0.0,0.536585,0.508772,0.456853,0.444444,0.557692,0.452736,0.285714,0.326531,...,0.224528,0.457143,0.442105,1.0,True,LAC,2021,0.0,DEN,2020-12-25
4,0.729996,0.5,0.560976,0.666667,0.375635,0.296296,0.384615,0.407960,0.523810,0.571429,...,0.405660,0.328571,0.578947,1.0,True,SAC,2021,1.0,PHO,2020-12-26
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7007,0.492084,0.0,0.317073,0.368421,0.302030,0.370370,0.442308,0.457711,0.238095,0.244898,...,0.407547,0.614286,0.600000,1.0,False,MIN,2025,NaN,None,None
7008,0.021110,0.0,0.195122,0.175439,0.284264,0.296296,0.365385,0.424544,0.523810,0.530612,...,0.345283,0.628571,0.463158,1.0,False,IND,2025,1.0,NYK,2025-05-31
7009,0.021252,0.0,0.536585,0.438596,0.510152,0.222222,0.346154,0.330017,0.309524,0.387755,...,0.179245,0.371429,0.284211,0.0,True,NYK,2025,0.0,IND,2025-05-31
7010,0.215091,0.0,0.463415,0.385965,0.467005,0.259259,0.403846,0.338308,0.357143,0.469388,...,0.349057,0.614286,0.610526,1.0,False,NYK,2025,NaN,None,None


In [25]:
full = df.merge(
    df[rolling_cols + ["team_opp_next", "date_next", "team"]],
    left_on=["team", "date_next"],
    right_on=["team_opp_next", "date_next"]
)

In [26]:
full

,Unnamed: 0,mp,fg,fga,fg%,3p,3pa,3p%,ft,fta,...,pts_max_opp_10_y,gmsc_max_opp_10_y,+/-_max_opp_10_y,total_opp_10_y,home_opp_10_y,won_10_y,team_10_y,season_10_y,team_opp_next_y,team_y
0,0.123378,0.0,0.390244,0.298246,0.446701,0.259259,0.346154,0.386401,0.523810,0.571429,...,0.169492,0.150943,0.457143,0.410526,1.0,False,DAL,2021,LAL,DAL
1,0.712594,0.0,0.365854,0.614035,0.205584,0.296296,0.423077,0.374793,0.309524,0.408163,...,0.355932,0.366038,0.542857,0.578947,1.0,False,MIL,2021,GSW,MIL
2,0.712737,0.0,0.487805,0.491228,0.416244,0.481481,0.461538,0.583748,0.571429,0.591837,...,0.389831,0.320755,0.571429,0.568421,0.0,True,BOS,2021,BRK,BOS
3,0.123235,0.0,0.536585,0.508772,0.456853,0.444444,0.557692,0.452736,0.285714,0.326531,...,0.169492,0.232075,0.371429,0.600000,0.0,False,DEN,2021,LAC,DEN
4,0.729996,0.5,0.560976,0.666667,0.375635,0.296296,0.384615,0.407960,0.523810,0.571429,...,0.338983,0.273585,0.457143,0.368421,0.0,True,PHO,2021,SAC,PHO
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6975,0.236486,0.0,0.634146,0.543860,0.538071,0.518519,0.500000,0.588723,0.333333,0.367347,...,0.474576,0.401887,0.371429,0.642105,0.0,False,MIN,2025,OKC,MIN
6976,0.100984,0.0,0.390244,0.315789,0.431472,0.370370,0.326923,0.583748,0.738095,0.734694,...,0.322034,0.296226,0.414286,0.568421,0.0,True,IND,2025,NYK,IND
6977,0.101127,0.0,0.560976,0.421053,0.553299,0.407407,0.403846,0.545605,0.595238,0.591837,...,0.338983,0.539623,0.542857,0.663158,1.0,False,NYK,2025,IND,NYK
6978,0.021110,0.0,0.195122,0.175439,0.284264,0.296296,0.365385,0.424544,0.523810,0.530612,...,0.186441,0.179245,0.371429,0.284211,0.0,True,NYK,2025,IND,NYK


## Re-running feature selection and the backtest on the enriched dataset

In [27]:
removed_columns = list(full.columns[full.dtypes == "object"]) + removed_columns
selected_columns = full.columns[~full.columns.isin(removed_columns)]

sfs.fit(full[selected_columns], full["target"])

,estimator,RidgeClassifier(alpha=1)
,n_features_to_select,30
,tol,None
,direction,'forward'
,scoring,None
,cv,TimeSeriesSpl...est_size=None)
,n_jobs,None
,alpha,1
,fit_intercept,True
,copy_X,True
,max_iter,None


In [28]:
predictors = list(selected_columns[sfs.get_support()])
predictors

['fg',
 'fga',
 'fg%',
 'fg_max',
 '3p_max',
 '3p%_max',
 'drb_max',
 'tov_max',
 'blk_opp',
 'fg%_max_opp',
 'fg_10_x',
 'fga_10_x',
 'fg%_10_x',
 'fg_max_10_x',
 '3p_max_10_x',
 '3p%_max_10_x',
 'drb_max_10_x',
 'blk_opp_10_x',
 'fg%_max_opp_10_x',
 'home_next',
 '3p_10_y',
 'orb_10_y',
 'pts_10_y',
 '3p%_max_10_y',
 'fta_max_10_y',
 'ft%_max_10_y',
 'gmsc_max_10_y',
 'total_10_y',
 'tov_opp_10_y',
 'stl_max_opp_10_y']

In [29]:
predictions = backtest(full, rr, predictors)

In [30]:
predictions = predictions[predictions["actual"] != 2]
accuracy_score(predictions["actual"], predictions["prediction"])

0.5748576078112286